Requirements

In [ ]:
%pip install -q kagglehub libreyolo
%pip install -q --upgrade jupyter ipywidgets
# !git clone https://github.com/LuisPeregrina/gdl-atsc-anti-spillback.git
# !mv gdl-atsc-anti-spillback/* .


Imports

In [ ]:
from pathlib import Path

import kagglehub
from libreyolo import LibreYOLO

from tools.mtid_split_yolo import split_dataset


Config

In [ ]:
MODEL_NAME = "LibreFOMOs-point"
DATASET_PATH = Path.cwd() / "dataset"
DATASET_NAME = "andreasmoegelmose/multiview-traffic-intersection-dataset"
EPOCHS = 2

# Configs per model
MODELS = {
    "LibreFOMOs-point": {
        "image_size": 96,
        "batch_size": -1,
    },
    "YOLOv9t": {
        "image_size": 640,
        "batch_size": -1,
    },
}
IMAGE_SIZE = MODELS[MODEL_NAME]["image_size"]
BATCH_SIZE = MODELS[MODEL_NAME]["batch_size"]


Dataset

In [ ]:

kagglehub.dataset_download(DATASET_NAME, output_dir=str(DATASET_PATH))
yaml_path = split_dataset(DATASET_PATH)
model = LibreYOLO(f"{MODEL_NAME}.pt")


Train

In [ ]:
model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
)
model.export(format="pt")

Validate

In [ ]:
metrics_val = model.val(data=yaml_path, split="val", batch=BATCH_SIZE)
print("Validation mAP50-95:", metrics_val["metrics/mAP50-95"])


Test

In [ ]:
# Test metrics (final evaluation)
metrics_test = model.val(data=yaml_path, split="test", batch=16)
print("Test mAP50-95:", metrics_test["metrics/mAP50-95"])

Results

In [ ]:
print("Validation metrics/mAP50-95:", metrics_val["metrics/mAP50-95"])
print("Validation speed/total_ms:", metrics_val["speed/total_ms"])
print("Test metrics/mAP50-95:", metrics_test["metrics/mAP50-95"])
print("Test speed/total_ms:", metrics_test["speed/total_ms"])